In [0]:
%pip install gspread google-auth
dbutils.library.restartPython()

In [ ]:
# Date/time range
dbutils.widgets.text("from_date", "17/07/2026", "From Date (dd/MM/yyyy)")
dbutils.widgets.text("from_time", "17:30", "From Time (HH:mm)")
dbutils.widgets.text("to_date", "17/07/2026", "To Date (dd/MM/yyyy)")
dbutils.widgets.text("to_time", "19:15", "To Time (HH:mm)")

REPORT_NAMES = [
    "D.Analysis - OSR PiE",
    "D.Analysis - OSR Topup",
    "D.Analysis - E3 Packing",
    "D.Analysis - Parcel Induct",
    "D.Analysis - Parcel Sortation",
    "D.Analysis - Inbound Decanting",
    "D.Analysis - OSR Decanting",
    "D.Analysis - BCR Inducting",
    "D.Analysis - E1/E2 Inducting",
]

# Filter mode + multi-select of report names
dbutils.widgets.dropdown("filter_mode", "All", ["All", "Include selected", "Exclude selected"], "Filter Mode")
dbutils.widgets.multiselect("selected_reports", REPORT_NAMES[0], REPORT_NAMES, "Select Reports")

# Where to send the backfill result. Default is a table shown in Databricks
# (nothing written to the sheet); choose "Google Sheet" to append to the Data
# tab and rebuild Processed Data (15mins), like the live job.
dbutils.widgets.dropdown("output_target", "Databricks table", ["Databricks table", "Google Sheet"], "Output Target")

from_date = dbutils.widgets.get("from_date")
from_time = dbutils.widgets.get("from_time")
to_date   = dbutils.widgets.get("to_date")
to_time   = dbutils.widgets.get("to_time")
filter_mode = dbutils.widgets.get("filter_mode")
selected = [s.strip() for s in dbutils.widgets.get("selected_reports").split(",") if s.strip()]
output_target = dbutils.widgets.get("output_target")

print(f"Backfill range: {from_date} {from_time} -> {to_date} {to_time} (UK local time)")
print(f"Filter: {filter_mode}" + (f" -> {selected}" if filter_mode != "All" else ""))
print(f"Output target: {output_target}")


In [0]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

uk = ZoneInfo("Europe/London")

start_local = datetime.strptime(f"{from_date} {from_time}", "%d/%m/%Y %H:%M").replace(tzinfo=uk)
end_local   = datetime.strptime(f"{to_date} {to_time}",   "%d/%m/%Y %H:%M").replace(tzinfo=uk)

# Validate alignment to 15-min boundaries
for label, t in [("From", start_local), ("To", end_local)]:
    if t.minute % 15 != 0:
        raise ValueError(f"{label} time {t.strftime('%H:%M')} is not on a 15-minute boundary (:00, :15, :30, :45)")

if end_local <= start_local:
    raise ValueError("To datetime must be after From datetime")

# Build each 15-min block as (start, end) pairs in UK local, plus UTC equivalents for filtering
blocks = []
cursor = start_local
while cursor < end_local:
    block_start_local = cursor
    block_end_local   = cursor + timedelta(minutes=15)
    blocks.append({
        "start_local": block_start_local,
        "end_local":   block_end_local,
        "start_utc":   block_start_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S"),
        "end_utc":     block_end_local.astimezone(ZoneInfo("UTC")).strftime("%Y-%m-%d %H:%M:%S"),
    })
    cursor = block_end_local

print(f"{len(blocks)} blocks to backfill:")
for b in blocks:
    print(f"  {b['start_local'].strftime('%d/%m/%Y %H:%M')} - {b['end_local'].strftime('%d/%m/%Y %H:%M')}")

In [0]:
import gspread
from google.oauth2.service_account import Credentials
import json

KEY_FILE_PATH = "/Workspace/Users/pakhei_tsang@next.co.uk/advance-mantis-398714-2168c9162641.json"  # adjust to your path

with open(KEY_FILE_PATH, "r") as f:
    creds_dict = json.load(f)

creds = Credentials.from_service_account_info(
    creds_dict, scopes=["https://www.googleapis.com/auth/spreadsheets"]
)
gc = gspread.authorize(creds)

SHEET_ID = "1uiv27J0YPDWqSVuN-QLpS-fOcfxAjewK5ajTd3aA8vI"
sh = gc.open_by_key(SHEET_ID)
ws = sh.worksheet("Data")

print("Connected to:", sh.title)

In [ ]:
df = (spark.read
      .format("delta")
      .load("abfss://landing@whsanalyticsdlsprodeuw.dfs.core.windows.net/streaming/landing_bonushub_event_parsed/delta/"))
df.createOrReplaceTempView("landing_bonus_hub_event_parsed")

all_reports = [
    {"name": "D.Analysis - OSR PiE",
     "area": "('Pick','Pie Station')", "event": "('COUN','MSKU','PSKU','RSIN')", "extra": ""},
    {"name": "D.Analysis - OSR Topup",
     "area": "('Topup','TopUp','PiOrQi')", "event": "('QSIN','TARS','TPUT')", "extra": ""},
    {"name": "D.Analysis - E3 Packing",
     "area": "('E3 - Packing')", "event": "('PackingParcelCompleteEvent','PackingItemScannedEvent')", "extra": ""},
    {"name": "D.Analysis - Parcel Induct",
     "area": "('Parcel Induct')", "event": "('APAR','PIND','SPAR','UPAR')", "extra": ""},
    {"name": "D.Analysis - Parcel Sortation",
     "area": None, "event": "('ParcelSortedToSack','SackMappedToPosition','SackUnMappedFromPosition')", "extra": ""},
    {"name": "D.Analysis - Inbound Decanting",
     "area": "('Inbound Decanting')", "event": "('DECN','TOPR')", "extra": ""},
    {"name": "D.Analysis - OSR Decanting",
     "area": "('OSR Decanting')", "event": "('ODEC','OTOP')", "extra": ""},
    {"name": "D.Analysis - BCR Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%RET%')"},
    {"name": "D.Analysis - E1/E2 Inducting",
     "area": "('Induct from E1/E2')", "event": "('SPOS')",
     "extra": "AND EXISTS(PAYLOAD_ATTRIBUTES, x -> x LIKE '%PIE4EDW%')"},
]

if filter_mode == "Include selected":
    reports = [r for r in all_reports if r['name'] in selected]
elif filter_mode == "Exclude selected":
    reports = [r for r in all_reports if r['name'] not in selected]
else:  # All
    reports = all_reports

if not reports:
    raise ValueError("Your filter selection matched no reports — check Filter Mode and Selected Reports")

print(f"Reports to run ({len(reports)} of {len(all_reports)}):")
for r in reports:
    print(f"  - {r['name']}")

COLUMNS = ['Date', 'Hour', 'PAYLOAD_BONUSCODE', 'PAYLOAD_EVENTTYPE',
           'Total_Quantity', 'Total_StandardHours', 'Total_SMV',
           'Week', 'Date Time Range', 'Report Name']

In [ ]:
# --- Std Hours divisor: read from Front!C2 of the linked spreadsheet (the same
#     value the Apps Script side used before processing moved into Databricks).
#     Accepts a percentage cell ("96.40%") or a bare number. `sh` is the
#     spreadsheet handle from the connection cell above. ---
_div_raw = (sh.worksheet("Front").acell("C2").value or "").strip()
if not _div_raw:
    raise ValueError("Front!C2 is empty - cannot determine the Std Hours divisor.")
divisor = float(_div_raw.rstrip('%')) / 100 if _div_raw.endswith('%') else float(_div_raw)
if divisor == 0:
    divisor = 1.0
print(f"Divisor (from Front!C2): {divisor}")

# --- Work-area pivot config: column order for Processed Data (15mins) D:L / N:V,
#     and each area's volume event type(s). Single source of truth. ---
PROC_AREAS = [
    {"report": "D.Analysis - OSR PiE",           "vol_events": {"MSKU", "PSKU"}},
    {"report": "D.Analysis - OSR Topup",         "vol_events": {"TPUT"}},
    {"report": "D.Analysis - E3 Packing",        "vol_events": {"PackingItemScannedEvent"}},
    {"report": "D.Analysis - Parcel Sortation",  "vol_events": {"ParcelSortedToSack"}},
    {"report": "D.Analysis - Parcel Induct",     "vol_events": {"SPAR"}},
    {"report": "D.Analysis - Inbound Decanting", "vol_events": {"DECN"}},
    {"report": "D.Analysis - OSR Decanting",     "vol_events": {"ODEC"}},
    {"report": "D.Analysis - BCR Inducting",     "vol_events": {"SPOS"}},
    {"report": "D.Analysis - E1/E2 Inducting",   "vol_events": {"SPOS"}},
]


In [ ]:
import pandas as pd
from datetime import datetime, timezone, date

# ============================================================
# CONSOLIDATED FETCH + OUTPUT
# One Delta scan for the WHOLE backfill (all blocks x selected reports),
# instead of len(blocks) x len(reports) separate spark.sql().toPandas() calls.
#
# Respects Filter Mode / Select Reports: the CASE/predicate array is built from
# `reports` (already filtered in the config cell), so only the selected reports
# are scanned, tagged and output. Using a SET (array + explode) rather than a
# first-match CASE preserves the original per-report semantics exactly - a row
# that satisfied two reports' filters fed both, and still does.
#
# The scan is bounded to [min start, max end] of the blocks for partition
# pruning, then filtered to the EXACT 15-min buckets (window_epoch IN ...).
#
# Output goes where the output_target widget says: "Google Sheet" appends to the
# Data tab in one batch (and the next cell rebuilds Processed Data); the default
# "Databricks table" writes nothing to the sheet and just displays the result.
# ============================================================

EPOCH_REF = date(2025, 12, 14)  # same week-numbering anchor as the live job
def _week_val(d):
    return ((d - EPOCH_REF).days // 7 + 46) % 52

def _blk_epoch(s):
    return int(datetime.strptime(s, "%Y-%m-%d %H:%M:%S").replace(tzinfo=timezone.utc).timestamp())

# 15-min bucket (UTC epoch) -> block metadata
blk_by_epoch = {}
for b in blocks:
    blk_by_epoch[_blk_epoch(b['start_utc'])] = {
        "week_val": _week_val(b['start_local'].date()),
        "date_time_range": f"{b['start_local'].strftime('%d/%m/%Y %H:%M')} - {b['end_local'].strftime('%d/%m/%Y %H:%M')}",
        "date_display": b['start_local'].strftime('%d/%m/%Y'),
    }

overall_start_utc = min(b['start_utc'] for b in blocks)
overall_end_utc = max(b['end_utc'] for b in blocks)
epoch_list = ", ".join(str(e) for e in sorted(blk_by_epoch))

# One report's {area, event, extra} config -> a standalone SQL boolean predicate
# (the `extra` field starts with "AND ", stripped so it can stand alone).
def _report_predicate(r):
    parts = []
    if r['area']:
        parts.append(f"PAYLOAD_AREACODE IN {r['area']}")
    parts.append(f"PAYLOAD_EVENTTYPE IN {r['event']}")
    extra = r['extra'].strip()
    if extra:
        if extra[:4].upper() == "AND ":
            extra = extra[4:].strip()
        parts.append(extra)
    return " AND ".join(f"({p})" for p in parts)

preds = [(_report_predicate(r), r['name']) for r in reports]
case_exprs = ",\n            ".join(f"CASE WHEN {pred} THEN '{name}' END" for pred, name in preds)
match_any = " OR ".join(f"({pred})" for pred, _ in preds)

query = f"""
  WITH base AS (
    SELECT
      date_format(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London'),'dd/MM/yyyy') AS Date,
      hour(from_utc_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP),'Europe/London')) AS Hour,
      CAST(FLOOR(unix_timestamp(to_timestamp(PAYLOAD_EVENTTIMESTAMP)) / 900) * 900 AS BIGINT) AS window_epoch,
      PAYLOAD_BONUSCODE AS bonus,
      PAYLOAD_EVENTTYPE AS eventtype,
      PAYLOAD_QUANTITY      AS qty,
      PAYLOAD_STANDARDHOURS AS std,
      PAYLOAD_SMV           AS smv,
      filter(array(
        {case_exprs}
      ), x -> x IS NOT NULL) AS report_names
    FROM landing_bonus_hub_event_parsed
    WHERE TRIM(PAYLOAD_WAREHOUSECODE) = 'X'
      AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) >= timestamp('{overall_start_utc}')
      AND to_timestamp(PAYLOAD_EVENTTIMESTAMP) <  timestamp('{overall_end_utc}')
      AND ( {match_any} )
  )
  SELECT
    Date, Hour, window_epoch, bonus, eventtype, report_name,
    SUM(qty) AS Total_Quantity,
    SUM(std) AS Total_StandardHours,
    SUM(smv) AS Total_SMV
  FROM base
  LATERAL VIEW explode(report_names) t AS report_name
  WHERE window_epoch IN ({epoch_list})
  GROUP BY Date, Hour, window_epoch, bonus, eventtype, report_name
"""

report_order = {r['name']: i for i, r in enumerate(reports)}

failures = []
total_rows_pushed = 0
pdf = None
try:
    pdf = spark.sql(query).toPandas()
    print(f"Consolidated query returned {len(pdf)} grouped rows "
          f"across {len(blocks)} block(s) x {len(reports)} report(s).")
except Exception as e:
    error_msg = f"{type(e).__name__}: {str(e)}"
    print(f"CONSOLIDATED QUERY FAILED - {error_msg}")
    failures.append({"block": f"{overall_start_utc}..{overall_end_utc} UTC",
                     "report": "(consolidated query)", "error": error_msg})

# --- Shape into the Data-tab COLUMNS order; note which blocks had data. ---
data_rows = []
blocks_with_data = set()
if pdf is not None:
    for _, row in pdf.iterrows():
        ep = int(row['window_epoch'])
        m = blk_by_epoch.get(ep)
        if m is None:
            continue  # safety: outside target buckets
        blocks_with_data.add(ep)
        hour_key = int(row['Hour']) if str(row['Hour']).strip().lstrip('-').isdigit() else 0
        data_rows.append({
            "sort": (ep, report_order.get(row['report_name'], 999), hour_key, str(row['bonus'])),
            "vals": [row['Date'], row['Hour'], row['bonus'], row['eventtype'],
                     row['Total_Quantity'], row['Total_StandardHours'], row['Total_SMV'],
                     m['week_val'], m['date_time_range'], row['report_name']],
        })
    data_rows.sort(key=lambda d: d['sort'])

empty_blocks = [(ep, m) for ep, m in blk_by_epoch.items() if ep not in blocks_with_data]

if output_target == "Google Sheet":
    # One placeholder per EMPTY block so the sheet marks it covered (matches the
    # live job). One per block is enough; the rebuild cell skips them (Hour == '').
    rows = list(data_rows)
    for ep, m in empty_blocks:
        rows.append({"sort": (ep, 999, 0, ''),
                     "vals": [m['date_display'], '', '', '(no data this window)', '', '', '',
                              m['week_val'], m['date_time_range'], '(no data this window)']})
    rows.sort(key=lambda d: d['sort'])
    values = [[str(c) for c in d['vals']] for d in rows]
    if pdf is not None and values:
        try:
            ws.append_rows(values, value_input_option='USER_ENTERED', table_range='A1')
            total_rows_pushed = len(values)
            print(f"Appended {total_rows_pushed} rows to the 'Data' tab in one batch.")
        except Exception as e:
            error_msg = f"{type(e).__name__}: {str(e)}"
            print(f"BATCH APPEND FAILED - {error_msg}")
            failures.append({"block": f"{len(blocks)} block(s)",
                             "report": "(sheet append)", "error": error_msg})
else:
    # Databricks-only (default): show the result as a table, write nothing to the sheet.
    result_pdf = pd.DataFrame([d['vals'] for d in data_rows], columns=COLUMNS)
    print("Output target = Databricks table (nothing written to Google Sheet).")
    print(f"{len(result_pdf)} data rows across {len(blocks) - len(empty_blocks)} block(s) with data.")
    if empty_blocks:
        print(f"{len(empty_blocks)} block(s) with no data:")
        for _, m in sorted(empty_blocks, key=lambda t: t[0]):
            print(f"  {m['date_time_range']}")
    display(result_pdf)

# --- Per-block x report breakdown for the run log. ---
if pdf is not None:
    counts = pdf.groupby(['window_epoch', 'report_name']).size()
    for ep in sorted(blk_by_epoch):
        m = blk_by_epoch[ep]
        print(f"\n=== Block: {m['date_time_range']} ===")
        any_rows = False
        for r in reports:
            n = int(counts.get((ep, r['name']), 0))
            if n:
                any_rows = True
                print(f"  [{r['name']}] {n} rows")
        if not any_rows:
            print("  (no data this block)")

print(f"\n--- Backfill summary ---")
print(f"Output target: {output_target}")
print(f"Blocks processed: {len(blocks)}")
print(f"Total rows pushed to sheet: {total_rows_pushed}")
print(f"Failures: {len(failures)}")
for f in failures:
    print(f"  {f['block']} | {f['report']} | {f['error']}")


In [ ]:
# ============================================================
# REBUILD "Processed Data (15mins)" from the Data tab
# Only runs when output_target == "Google Sheet" (i.e. this backfill actually
# wrote rows to the Data tab). In "Databricks table" mode nothing was written to
# the sheet, so there is nothing to rebuild and this cell is skipped.
#
# Runs once after ALL blocks are appended above, so the rebuild reflects the
# full backfilled range. Pivot: group raw Data rows by (Date Time Range, Hour, Bonus).
#   D:L = sum(StandardHours) per area / divisor ; M = total ;
#   N:V = sum(Quantity) per area for that area's volume event(s).
# Databricks is the sole writer of this tab (Apps Script no longer processes).
# ============================================================
if output_target != "Google Sheet":
    print("Output target = Databricks table; skipping 'Processed Data (15mins)' sheet rebuild.")
else:
    from datetime import datetime
    from collections import OrderedDict

    def _f(x):
        try:
            return float(str(x).replace(',', '').strip())
        except Exception:
            return 0.0

    def _parse_start(dtr):
        try:
            return datetime.strptime(dtr.split(' - ')[0].strip(), '%d/%m/%Y %H:%M')
        except Exception:
            return datetime.max

    area_index = {a["report"]: i for i, a in enumerate(PROC_AREAS)}
    n_areas = len(PROC_AREAS)

    all_data = ws.get_all_values()            # ws = Data worksheet (from earlier cell)
    data_rows = all_data[1:] if all_data else []

    # Data cols: A=Date B=Hour C=Bonus D=EventType E=Qty F=StdHours G=SMV H=Week I=DateTimeRange J=ReportName
    groups = OrderedDict()
    for r in data_rows:
        if len(r) < 10:
            continue
        hour = r[1].strip()
        if hour == '':                        # skip "(no data this window)" placeholders
            continue
        ai = area_index.get(r[9].strip())     # Report Name
        if ai is None:
            continue
        key = (r[8].strip(), hour, r[2].strip())   # (Date Time Range, Hour, Bonus)
        g = groups.get(key)
        if g is None:
            g = {"std": [0.0] * n_areas, "vol": [0.0] * n_areas}
            groups[key] = g
        g["std"][ai] += _f(r[5])              # Total_StandardHours (all event types)
        if r[3].strip() in PROC_AREAS[ai]["vol_events"]:
            g["vol"][ai] += _f(r[4])          # Total_Quantity (volume event only)

    out = []
    for (dtr, hour, bonus), g in groups.items():
        std = [s / divisor for s in g["std"]]
        row = [dtr, hour, bonus] + std + [sum(std)] + g["vol"]
        out.append((_parse_start(dtr), bonus, row))

    out.sort(key=lambda t: (t[0], t[1]))
    proc_values = [t[2] for t in out]

    proc_ws = sh.worksheet("Processed Data (15mins)")
    proc_ws.batch_clear(["A2:V"])
    if proc_values:
        proc_ws.update("A2", proc_values, value_input_option="USER_ENTERED")

    print(f"Processed Data (15mins) rebuilt: {len(proc_values)} rows")
